# DiD-BCF paper rerun: main_wave_02_shard_04

This independent notebook runs 82 complete fits using the pinned repaired engine, GFR 50, burn-in 0, 500 retained draws per chain, thinning 5, and three chains. It preserves the original panels, priors and seed schedule.

The estimated runtime is 8.0 hours with two CPU workers, including setup. Runtime varies across Colab machines. The session stops before 9.5 hours; rerun this same notebook with its checkpoint if any fits remain. CPU runtime, Python 3.12 to 3.14, and approximately 12 GB RAM are appropriate. The controller uses one worker when RAM or CPU quotas require it.

Download contains paper evaluation summaries, small scalar diagnostics and provenance. Posterior arrays, sweep traces and native binaries stay on the runtime disk. Completed fits are recoverable; an interrupted fit restarts from its original seeds. No fits are excluded based on diagnostics or performance.

The earlier budget was selected for paper performance. It did not pass the previous full-state diagnostic gate and does not certify nominal interval coverage.


In [ ]:
import os, sys, subprocess, time
from pathlib import Path
assert (3, 12) <= sys.version_info[:2] <= (3, 14), "Use Python 3.12 to 3.14"
SESSION_STARTED = time.time()
DEADLINE = SESSION_STARTED + 9.5 * 3600
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"
SOURCE_COMMIT = '3078e2ca1155b932feab92c722d697803370b8c7'
REPO = Path("/content/didbcf_paper_repo") if Path("/content").exists() else Path.cwd() / "didbcf_paper_repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/hugogobato/DiD-BCF.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", SOURCE_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", 'Simulation_Studies_Revision/Proper_Prior_Rerun/Paper_Rerun_20261011'], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == SOURCE_COMMIT
PACKAGE = REPO / 'Simulation_Studies_Revision/Proper_Prior_Rerun/Paper_Rerun_20261011'
import json
manifest = json.loads((PACKAGE / "manifest.json").read_text())
requirements = [name + "==" + version for name, version in manifest["dependencies"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *requirements], check=True)
HELPERS = PACKAGE / "helpers"
print("Pinned sources and dependencies installed:", SOURCE_COMMIT)


In [ ]:
# Optional: preserve checkpoints in your own Drive folder.
USE_DRIVE = False
RESUME_ZIP = ""  # Path to this notebook's previously downloaded metrics ZIP.
WORKERS = 2
NAME = 'main_wave_02_shard_04'
WORK = Path("/content") / (NAME + "_work") if Path("/content").exists() else Path.cwd() / (NAME + "_work")
OUT = WORK / "paper_output"
OUTPUT_FILE = WORK / ("didbcf_paper_" + NAME + ".zip")
ASSIGNMENT = PACKAGE / "assignments" / (NAME + ".json")
DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DiD_BCF_Paper_Rerun_20261011")
    previous = DRIVE_DIR / OUTPUT_FILE.name
    if not RESUME_ZIP and previous.exists():
        RESUME_ZIP = str(previous)
# To restore a local downloaded ZIP, upload it in Colab's Files pane and set
# RESUME_ZIP to its /content/... path before running the next cell.
print("Fits:", 82, "Output:", OUTPUT_FILE)


In [ ]:
command = [sys.executable, str(HELPERS / "runtime.py"), "prepare",
           "--assignment", str(ASSIGNMENT), "--work", str(WORK), "--out", str(OUT),
           "--cache", str(WORK / "source_cache"), "--deadline", str(DEADLINE)]
if RESUME_ZIP:
    command += ["--resume", RESUME_ZIP]
subprocess.run(command, check=True)


In [ ]:
command = [sys.executable, str(HELPERS / "runtime.py"), "run",
           "--assignment", str(ASSIGNMENT), "--work", str(WORK), "--out", str(OUT),
           "--setup", str(WORK / "setup_complete.json"), "--zip", str(OUTPUT_FILE),
           "--workers", str(WORKERS), "--deadline", str(DEADLINE)]
if DRIVE_DIR:
    command += ["--drive-dir", str(DRIVE_DIR)]
result = subprocess.run(command)
if result.returncode not in (0, 2):
    raise RuntimeError("Execution failed; inspect the job logs and preserve any metrics checkpoint")
status = json.loads((OUT / "bundle/status.json").read_text())
print("Completed:", status["completed"], "/", status["expected"])
if status["missing_job_ids"]:
    print("Resume this same notebook with its ZIP to complete", len(status["missing_job_ids"]), "remaining fits.")


In [ ]:
output_file = str(OUTPUT_FILE)
assert Path(output_file).is_file(), "No completed metrics archive was created"
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
